In [1]:
!pip install --no-cache-dir --force-reinstall "tinker-cookbook @ git+https://github.com/thinking-machines-lab/tinker-cookbook.git@nightly"

  Cloning https://github.com/thinking-machines-lab/tinker-cookbook.git (to revision nightly) to /tmp/pip-install-4bz3p0dm/tinker-cookbook_bec006b4c27143cf9c76762e8db1ab02
  Running command git clone --filter=blob:none --quiet https://github.com/thinking-machines-lab/tinker-cookbook.git /tmp/pip-install-4bz3p0dm/tinker-cookbook_bec006b4c27143cf9c76762e8db1ab02
  Running command git checkout -b nightly --track origin/nightly
  Switched to a new branch 'nightly'
  Branch 'nightly' set up to track remote branch 'nightly' from 'origin'.
  Resolved https://github.com/thinking-machines-lab/tinker-cookbook.git to commit 839c27d21b66d3027c6866238e6e3f02406c90ed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 36.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 217.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
import tinker_cookbook.weights._adapter as A

FORCED_FUSED_RANK = 32

def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int, comp_slices: list, component_order: list):
    delta = B.float() @ A_mat.float()

    # THIẾT LẬP TỶ LỆ TOÁN HỌC (GRID SEARCH)
    w_q = 1.0
    w_k = 1.0
    w_v = 1.8  # V-Heavy (Ưu tiên nén giữ lại ngữ nghĩa)
    
    # 1. Nhân tỷ lệ ưu tiên trước khi SVD
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        comp_name = component_order[i]
        w = 1.0
        if 'q' in comp_name.lower(): w = w_q
        elif 'k' in comp_name.lower(): w = w_k
        elif 'v' in comp_name.lower(): w = w_v
        delta[row_start:row_end, :] *= w

    # 2. Chạy SVD nén xuống Rank 32
    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    U = U[:, :rank]
    S = S[:rank]
    Vh = Vh[:rank, :]

    sroot = torch.sqrt(S)
    B_new = U * sroot.unsqueeze(0)
    A_new = sroot.unsqueeze(1) * Vh

    # 3. Trả lại tỷ lệ gốc sau khi nén
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        comp_name = component_order[i]
        w = 1.0
        if 'q' in comp_name.lower(): w = w_q
        elif 'k' in comp_name.lower(): w = w_k
        elif 'v' in comp_name.lower(): w = w_v
        B_new[row_start:row_end, :] /= w

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous()


def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    assert component_order is not None

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []   
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(f"Missing component {comp_name!r}")
        lora_A, lora_B = comp_by_name[comp_name]
        r = lora_A.shape[0]
        out_dim = lora_B.shape[0]

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim, merged_rank, dtype=merged_lora_A.dtype, device=merged_lora_A.device
    )

    rank_offset = 0
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        _, lora_B = comp_by_name[component_order[i]]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    final_rank = merged_rank
    if merged_rank > FORCED_FUSED_RANK:
        # Gọi hàm SVD đặc chế có chứa comp_slices
        merged_lora_B, merged_lora_A = _compress_lora_pair_to_rank(
            merged_lora_B, merged_lora_A, FORCED_FUSED_RANK, comp_slices, component_order
        )
        final_rank = FORCED_FUSED_RANK

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"
    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank

# Kích hoạt Monkey-patch
A._merge_fused_projections = patched_merge_fused_projections
print("✅ Đã vá lỗi! C-W SVD Patch (V-Heavy) đã sẵn sàng.")

✅ Đã vá lỗi! C-W SVD Patch (V-Heavy) đã sẵn sàng.


In [3]:
import os
from tinker_cookbook import weights

input_adapter_path = "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20/"
output_path = "/kaggle/working/nemotron-adapter-ready-to-submit"

print("Đang nén Adapter bằng C-W SVD... Quá trình này có thể mất vài phút.")

weights.build_lora_adapter(
    base_model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    adapter_path=input_adapter_path,
    output_path=output_path,
)

os.system(f"zip -r submission.zip {output_path}")
print("🎉 Đã tạo xong file submission.zip! Hãy refresh cột Output bên phải để tải về.")

Đang nén Adapter bằng C-W SVD... Quá trình này có thể mất vài phút.


Fetching 35 files:   0%|          | 0/35 [00:00<?, ?it/s]

MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


  adding: kaggle/working/nemotron-adapter-ready-to-submit/ (stored 0%)
  adding: kaggle/working/nemotron-adapter-ready-to-submit/adapter_config.json (deflated 49%)
  adding: kaggle/working/nemotron-adapter-ready-to-submit/adapter_model.safetensors (deflated 8%)
🎉 Đã tạo xong file submission.zip! Hãy refresh cột Output bên phải để tải về.
